# Tutorial 12: Universal Graph Neural Networks for Quantum Circuit Design

## Why Graph ML?

Tutorial 8 trained a tabular DNN: design parameters → Hamiltonian targets. This works for a **fixed** topology but breaks if you change the circuit structure.

The **Universal GNN** replaces this with a **heterogeneous graph** of geometric embeddings:
1. Design parameters → `build_layout()` → Shapely polygons
2. Components → **static embedding** = `param_sum ∥ geometric_moments ∥ shape_tensor`
3. Connections → **typed physical edges** with coupling/overlap geometry
4. Full layout → **virtual hub node** for global context
5. **HeteroConv GNN** learns which design parameters affect which Hamiltonian targets

### Target assignment
- **Node targets**: `qubit_freq`, `anharmonicity`, `cavity_freq` — intrinsic to components
- **Edge targets**: `g` (coupling strength), `kappa` (resonator linewidth) — arise from interactions

During training, ALL nodes get ALL node targets and ALL edges get ALL edge targets. The GNN discovers the correlations.
At inference, we use a **readout map** to extract predictions from the correct nodes/edges.


In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import os, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from sklearn.manifold import TSNE
from sklearn.metrics import r2_score
from torch_geometric.loader import DataLoader

from squadds.ml.universal.geometry.layout import build_layout
from squadds.ml.universal.geometry.viz import plot_layout, plot_component
from squadds.ml.universal.features.node_encoder import (
    compute_static_embedding, get_polygon_for_component,
    static_embedding_dim, DEFAULT_SHAPE_RESOLUTION,
)
from squadds.ml.universal.features.moments import compute_moments, moment_names
from squadds.ml.universal.features.edge_extractor import EdgeFeatureExtractor, edge_feature_dim
from squadds.ml.universal.graph.netlist import CircuitNetlist, ComponentSpec, EdgeSpec
from squadds.ml.universal.graph.builder import UniversalGraphBuilder
from squadds.ml.universal.graph.virtual_hub import _rasterize_in_bounds, spatial_edge_feature_dim
from squadds.ml.universal.model.gat_model import (
    UniversalGNN, NODE_TARGET_NAMES, EDGE_TARGET_NAMES,
    NODE_INFERENCE_READOUT, EDGE_INFERENCE_READOUT,
)
from squadds.ml.universal.trainer import UniversalTrainer

SHAPE_RES = DEFAULT_SHAPE_RESOLUTION
print(f"Shape resolution: {SHAPE_RES}x{SHAPE_RES}")
print(f"Node embedding dim: {static_embedding_dim(SHAPE_RES)}")
print(f"Edge feature dim: {edge_feature_dim(SHAPE_RES)}")
print(f"Node targets: {NODE_TARGET_NAMES}")
print(f"Edge targets: {EDGE_TARGET_NAMES}")

seed = 42
torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)


---\n## 1. From Design Parameters to Physical Layout

In [ ]:
df = pd.read_parquet("data/training_data.parquet").drop_duplicates().reset_index(drop=True)
print(f"Dataset: {df.shape[0]:,} rows x {df.shape[1]} columns")
df.head(3)


In [ ]:
row = df.iloc[0]
lyt = build_layout(
    cross_length=row["cross_length"], cross_gap=row["cross_gap"],
    claw_length=row["claw_length"], ground_spacing=row["ground_spacing"],
    coupling_length=row["coupling_length"], total_length=row["total_length"],
)
fig = plot_layout(lyt)
plt.suptitle(f"Layout: cross_length={row['cross_length']}, total_length={row['total_length']}", y=1.01)
plt.show()


---\n## 2. Static Embedding Deep Dive

In [ ]:
comp_names = ["qubit", "claw", "resonator", "feedline"]
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i, name in enumerate(comp_names):
    poly = get_polygon_for_component(lyt[name])
    params = lyt[name].get("params", {})
    emb = compute_static_embedding(poly, params=params, shape_resolution=SHAPE_RES)
    mom = compute_moments(poly)
    print(f"=== {name.upper()} ===  param_sum={sum(params.values()) if params else 0:.1f}")
    for mn, mv in zip(moment_names(), mom): print(f"  {mn:20s}: {mv:12.2f}")
    print()
    axes[0, i].imshow(emb[9:].reshape(SHAPE_RES, SHAPE_RES), cmap='viridis', interpolation='nearest')
    axes[0, i].set_title(f"{name.title()} Shape"); axes[0, i].axis('off')
    axes[1, i].barh(moment_names(), mom, color='steelblue')
    axes[1, i].set_title(f"{name.title()} Moments"); axes[1, i].tick_params(labelsize=7)
plt.suptitle("Static Embeddings", fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()


### Embedding Space (t-SNE from 200 dataset samples)

In [ ]:
embs, labels = [], []
for _, r in df.head(200).iterrows():
    ly = build_layout(cross_length=r["cross_length"], cross_gap=r["cross_gap"],
        claw_length=r["claw_length"], ground_spacing=r["ground_spacing"],
        coupling_length=r["coupling_length"], total_length=r["total_length"])
    for cn in comp_names:
        embs.append(compute_static_embedding(get_polygon_for_component(ly[cn]),
            params=ly[cn].get("params",{}), shape_resolution=SHAPE_RES))
        labels.append(cn.title())
X_2d = TSNE(perplexity=30, random_state=42).fit_transform(np.array(embs))
plt.figure(figsize=(10,7))
sns.scatterplot(x=X_2d[:,0], y=X_2d[:,1], hue=labels, palette="deep", s=60, alpha=0.8)
plt.title("t-SNE of Component Embeddings"); plt.grid(alpha=0.3); plt.show()


---
## 3. Heterogeneous Graph Assembly

| Nodes | Edges |
|---|---|
| `component` (static embedding) | `physical` (component ↔ component): coupling + overlap geometry |
| `virtual` (full layout embedding) | `spatial_{in,out}` (component ↔ virtual): rel pos + area/perim fractions |

**Node targets** (ALL nodes predict ALL during training):
| Target | Physically from | Read at inference from |
|---|---|---|
| `qubit_freq_GHz` | TransmonCross | TransmonCross node |
| `anharmonicity_MHz` | TransmonCross | TransmonCross node |
| `cavity_freq_GHz` | RouteMeander | RouteMeander node |

**Edge targets** (ALL edges predict ALL during training):
| Target | Physically from | Read at inference from |
|---|---|---|
| `g_MHz` | qubit-claw interaction | TransmonCross↔Claw edge |
| `kappa_kHz` | resonator-feedline interaction | RouteMeander↔CoupledLineTee edge |


In [ ]:
netlist = CircuitNetlist(
    components=[
        ComponentSpec(name="qubit", component_type="TransmonCross"),
        ComponentSpec(name="claw", component_type="Claw"),
        ComponentSpec(name="resonator", component_type="RouteMeander"),
        ComponentSpec(name="feedline", component_type="CoupledLineTee"),
    ],
    edges=[
        EdgeSpec(src="qubit", dst="claw", coupling_type="capacitive"),
        EdgeSpec(src="claw", dst="resonator", coupling_type="galvanic"),
        EdgeSpec(src="resonator", dst="feedline", coupling_type="capacitive"),
    ],
)

builder = UniversalGraphBuilder(shape_resolution=SHAPE_RES, cache_dir="graph_cache")
data = builder.build(lyt, netlist, global_features={"dielectric_constant": 11.45})
print("=== HeteroData Graph ===")
print(data)

print("\nNode inference readout:")
for name, ctype, readout in zip(
    data['component'].component_name, data['component'].component_type,
    data['component'].inference_readout):
    print(f"  {name:12s} ({ctype:16s}): {readout if readout else '(passive)'}")

print("\nEdge inference readout:")
for i, (src_t, dst_t) in enumerate(data['component','physical','component'].edge_component_types):
    key = (src_t, dst_t)
    readout = EDGE_INFERENCE_READOUT.get(key, [])
    if readout:
        print(f"  edge {i}: {src_t} <-> {dst_t} -> reads {readout}")


In [ ]:
# Edge overlap + hub masked shapes
edge_extractor = EdgeFeatureExtractor(shape_resolution=SHAPE_RES)
epairs = [("qubit","claw","cap"),("claw","resonator","galv"),("resonator","feedline","cap")]
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for i, (s, d, _) in enumerate(epairs):
    feat = edge_extractor.extract(get_polygon_for_component(lyt[s]),
        get_polygon_for_component(lyt[d]), coupling_type=["capacitive","galvanic","capacitive"][i])
    axes[i].imshow(feat[8:].reshape(SHAPE_RES, SHAPE_RES), cmap='hot', interpolation='nearest')
    axes[i].set_title(f"{s} <-> {d}"); axes[i].axis('off')
plt.suptitle("Physical Edge: Overlap Shape Tensors", fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()


---
## 4. Training

Each row → HeteroData graph. ALL nodes get ALL 3 node targets. ALL edges get ALL 2 edge targets.
The GNN learns the physics through message passing.


In [ ]:
N_SAMPLES = 5000  # Increase for production

NODE_SCALES = {"qubit_frequency_GHz": 1.0, "anharmonicity_MHz": 100.0, "cavity_frequency_GHz": 1.0}
EDGE_SCALES = {"g_MHz": 100.0, "kappa_kHz": 100.0}

df_sub = df.head(N_SAMPLES)
graph_dataset = []
builder = UniversalGraphBuilder(shape_resolution=SHAPE_RES, cache_dir="graph_cache")

print(f"Building {N_SAMPLES} graphs...")
for idx, row in df_sub.iterrows():
    lyt_i = build_layout(
        cross_length=row["cross_length"], cross_gap=row["cross_gap"],
        claw_length=row["claw_length"], ground_spacing=row["ground_spacing"],
        coupling_length=row["coupling_length"], total_length=row["total_length"],
    )
    data_i = builder.build(lyt_i, netlist, global_features={"dielectric_constant": 11.45})
    
    # ALL nodes get ALL node targets
    y_n = data_i["component"].y.clone()
    for i in range(y_n.size(0)):
        y_n[i, 0] = row["qubit_frequency_GHz"] / NODE_SCALES["qubit_frequency_GHz"]
        y_n[i, 1] = row["anharmonicity_MHz"] / NODE_SCALES["anharmonicity_MHz"]
        y_n[i, 2] = row["cavity_frequency_GHz"] / NODE_SCALES["cavity_frequency_GHz"]
    data_i["component"].y = y_n
    
    # ALL edges get ALL edge targets
    y_e = data_i["component", "physical", "component"].y.clone()
    for i in range(y_e.size(0)):
        y_e[i, 0] = row["g_MHz"] / EDGE_SCALES["g_MHz"]
        y_e[i, 1] = row["kappa_kHz"] / EDGE_SCALES["kappa_kHz"]
    data_i["component", "physical", "component"].y = y_e
    
    graph_dataset.append(data_i)
    if (idx + 1) % 1000 == 0:
        print(f"  {idx+1}/{N_SAMPLES}")

print(f"Done: {len(graph_dataset)} graphs")
print(f"Node targets shape: {graph_dataset[0]['component'].y.shape} (all nodes, 3 targets)")
print(f"Edge targets shape: {graph_dataset[0]['component','physical','component'].y.shape} (all edges, 2 targets)")


In [ ]:
split = int(0.85 * len(graph_dataset))
train_loader = DataLoader(graph_dataset[:split], batch_size=32, shuffle=True)
val_loader = DataLoader(graph_dataset[split:], batch_size=32)

s = graph_dataset[0]
model = UniversalGNN(
    comp_dim=s["component"].x.size(1),
    virt_dim=s["virtual"].x.size(1),
    phys_edge_dim=s["component","physical","component"].edge_attr.size(1),
    spat_edge_dim=s["component","spatial_in","virtual"].edge_attr.size(1),
    hidden_dim=128, num_layers=3, num_heads=4, edge_hidden=32,
)
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

trainer = UniversalTrainer(model, learning_rate=1e-3, checkpoint_dir="checkpoints")
history = trainer.train_loop(train_loader, val_loader, epochs=500, patience=50)
trainer.load_checkpoint("best_model.pt")
print("Best model loaded.")


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(history["train_loss"], label="Train"); ax1.plot(history["val_loss"], label="Val")
ax1.set(xlabel="Epoch", ylabel="Loss", title="Total Loss"); ax1.legend(); ax1.grid(alpha=0.3)
ax2.plot(history["train_node"], label="Node Train"); ax2.plot(history["val_node"], label="Node Val")
ax2.plot(history["train_edge"], label="Edge Train", ls='--')
ax2.plot(history["val_edge"], label="Edge Val", ls='--')
ax2.set(xlabel="Epoch", ylabel="Loss", title="Node vs Edge Loss"); ax2.legend(); ax2.grid(alpha=0.3)
plt.tight_layout(); plt.show()


---\n## 5. Parity Plots

In [ ]:
model.eval()
all_yn, all_ynp, all_ye, all_yep = [], [], [], []
with torch.no_grad():
    for batch in val_loader:
        out = model(batch)
        all_yn.append(batch["component"].y)
        all_ynp.append(out["node_preds"])
        all_ye.append(batch["component","physical","component"].y)
        all_yep.append(out["edge_preds"])

yn_t = torch.cat(all_yn).numpy(); yn_p = torch.cat(all_ynp).numpy()
ye_t = torch.cat(all_ye).numpy(); ye_p = torch.cat(all_yep).numpy()

targets = [
    ("Qubit Freq (GHz)", yn_t, yn_p, 0, 1.0),
    ("Anharmonicity (MHz)", yn_t, yn_p, 1, 100.0),
    ("Cavity Freq (GHz)", yn_t, yn_p, 2, 1.0),
    ("g (MHz)", ye_t, ye_p, 0, 100.0),
    ("Kappa (kHz)", ye_t, ye_p, 1, 100.0),
]

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()
for i, (name, yt, yp, idx, sc) in enumerate(targets):
    t, p = yt[:, idx] * sc, yp[:, idx] * sc
    axes[i].scatter(t, p, alpha=0.3, s=8, color='crimson')
    lo, hi = min(t.min(), p.min()), max(t.max(), p.max())
    if lo != hi:
        axes[i].plot([lo, hi], [lo, hi], 'k--', lw=2)
        axes[i].text(0.05, 0.9, f"R² = {r2_score(t, p):.3f}", transform=axes[i].transAxes,
                    fontsize=12, bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
    axes[i].set(title=name, xlabel="True", ylabel="Predicted"); axes[i].grid(alpha=0.3)
axes[-1].axis('off')
plt.suptitle("Parity Plots: Node & Edge Targets", fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()


---
## 6. Scale Invariance: The Holy Grail

The model was trained on **qubit-claw-resonator-feedline** graphs. Because it uses heterogeneous GNN on geometric embeddings, it can handle **different topologies at inference**.

### Case 1: Qubit-Claw Only (reduced topology)


In [ ]:
test_row = df.iloc[-1]
lyt_test = build_layout(
    cross_length=test_row["cross_length"], cross_gap=test_row["cross_gap"],
    claw_length=test_row["claw_length"], ground_spacing=test_row["ground_spacing"],
    coupling_length=test_row["coupling_length"], total_length=test_row["total_length"],
)

# --- Layout visualization ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
# Full layout for reference
plot_layout(lyt_test, ax=ax1)
ax1.set_title("Full Layout (reference)", fontweight='bold')

# Reduced layout
plot_component(lyt_test["qubit"], "qubit", ax=ax2, show_etch=False)
plot_component(lyt_test["claw"], "claw", ax=ax2, show_etch=False)
ax2.autoscale_view()
ax2.set_title("Case 1: Qubit-Claw Only", fontweight='bold')
ax2.set_aspect('equal')
plt.tight_layout(); plt.show()


In [ ]:
reduced_netlist = CircuitNetlist(
    components=[
        ComponentSpec(name="qubit", component_type="TransmonCross"),
        ComponentSpec(name="claw", component_type="Claw"),
    ],
    edges=[EdgeSpec(src="qubit", dst="claw", coupling_type="capacitive")],
)

data_r = builder.build(lyt_test, reduced_netlist, global_features={"dielectric_constant": 11.45})
model.eval()
with torch.no_grad():
    out_r = model(data_r)

node_scales = [1.0, 100.0, 1.0]
edge_scales = [100.0, 100.0]

def read_edge_predictions(data, out, edge_scales):
    # Average both directions of each undirected edge and show readout.
    ei = data['component','physical','component'].edge_index
    etypes = data['component','physical','component'].edge_component_types
    ep = out['edge_preds']
    names = data['component'].component_name
    
    # Group edges by undirected pair (min_idx, max_idx)
    seen = {}
    for i in range(ei.size(1)):
        s, d = ei[0, i].item(), ei[1, i].item()
        key = (min(s, d), max(s, d))
        if key not in seen:
            seen[key] = []
        seen[key].append(i)
    
    results = []
    for (a, b), indices in seen.items():
        avg_pred = sum(ep[idx] for idx in indices) / len(indices)
        st = data['component'].component_type[a]
        dt = data['component'].component_type[b]
        # Check both orderings for readout
        readout = EDGE_INFERENCE_READOUT.get((st, dt), []) or EDGE_INFERENCE_READOUT.get((dt, st), [])
        if readout:
            for j, tname in enumerate(EDGE_TARGET_NAMES):
                if tname in readout:
                    results.append(f"  {names[a]} <-> {names[b]} -> {tname}: {avg_pred[j].item() * edge_scales[j]:.3f}")
    return results

print("=== Case 1: Qubit-Claw Only ===")
print(f"Graph: {data_r['component'].x.size(0)} components, "
      f"{data_r['component','physical','component'].edge_index.size(1)//2} physical edges\n")

print("Node predictions (inference readout):")
for i, (name, ctype) in enumerate(zip(data_r['component'].component_name, data_r['component'].component_type)):
    readout = NODE_INFERENCE_READOUT.get(ctype, [])
    preds = out_r['node_preds'][i]
    for j, tname in enumerate(NODE_TARGET_NAMES):
        if tname in readout:
            print(f"  {name:12s} -> {tname}: {preds[j].item() * node_scales[j]:.3f}")

print("\nEdge predictions (inference readout):")
for line in read_edge_predictions(data_r, out_r, edge_scales):
    print(line)

print(f"\nGround truth: qubit_freq={test_row['qubit_frequency_GHz']:.3f}, "
      f"anharmonicity={test_row['anharmonicity_MHz']:.2f}, g={test_row['g_MHz']:.2f}")
print("Note: no cavity_freq/kappa readout since no resonator/feedline in topology.")


### Case 2: Extended Topology (6 components)

Add a second feedline and resonator pair. The model automatically provides:
- `cavity_freq` for resonator2
- `kappa` for the resonator2-feedline2 edge


In [ ]:
from shapely.affinity import translate, scale as shp_scale
from shapely.ops import unary_union
from shapely.geometry import box
from squadds.ml.universal.features.node_encoder import get_polygon_for_component

# ── Build extended geometry via affine transforms ─────────────────────────────
# feedline1: x≈[-6,6], y=[1100,1300]  (center y=1200)
# feedline2: same x, shifted down 400 um → y=[700,900]
# Combined feedline: single bar from y=700 to y=1300 (looks galvanically joined)
dy = -400

def shift_comp(comp_dict, dx=0, dy=0):
    out = {}
    for k, v in comp_dict.items():
        try:
            out[k] = translate(v, xoff=dx, yoff=dy)
        except Exception:
            out[k] = v
    return out

fl_poly  = lyt_test["feedline"].get("trace") or lyt_test["feedline"].get("polygon")
fl2_poly = translate(fl_poly, yoff=dy)

# Merged feedline: union + fill gap between them → one solid bar
fl_merged = unary_union([fl_poly, fl2_poly,
                         box(fl_poly.bounds[0], fl2_poly.bounds[3],
                             fl_poly.bounds[2], fl_poly.bounds[1])])

# resonator2: shift resonator down + mirror horizontally → goes rightward
res_poly  = get_polygon_for_component(lyt_test["resonator"])
res2_poly = translate(shp_scale(res_poly, xfact=-1, yfact=1, origin=(0, 0)), yoff=dy)

# Build lyt_ext for GNN cell below
lyt_ext = dict(lyt_test)
lyt_ext["feedline1"] = lyt_test["feedline"]
lyt_ext["resonator1"] = lyt_test["resonator"]
fl2_dict = shift_comp(lyt_test["feedline"], dy=dy)
lyt_ext["feedline2"]  = fl2_dict
lyt_ext["resonator2"] = {"polygon": res2_poly,
                          "params": lyt_test["resonator"].get("params", {})}

# ── Visualization ──────────────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 8))

# Left: training topology
plot_layout(lyt_test, ax=ax1)
ax1.set_title("Training Topology (4 components)\nqubit → claw → resonator → feedline",
    fontweight="bold")

# Right: extended topology
plot_component(lyt_test["qubit"],    "qubit",     ax=ax2, show_etch=False)
plot_component(lyt_test["claw"],     "claw",      ax=ax2, show_etch=False)
plot_component(lyt_test["resonator"],"resonator1",ax=ax2, show_etch=False)

# Single merged feedline bar (prime_start at top, prime_end at bottom)
ax2.fill(*fl_merged.exterior.xy, color="#9b59b6", alpha=0.85, zorder=3)
ax2.plot(*fl_merged.exterior.xy, color="#6c3483", lw=1.5, zorder=4)
# prime_start / prime_end dots
top_y  = fl_poly.bounds[3]
bot_y  = fl2_poly.bounds[1]
ax2.plot(0, top_y, "o", color="#6c3483", ms=5, zorder=5)
ax2.plot(0, bot_y, "o", color="#6c3483", ms=5, zorder=5)
ax2.text(18, top_y, "prime_start", va="center", fontsize=7, color="#6c3483")
ax2.text(18, bot_y, "prime_end",   va="center", fontsize=7, color="#6c3483")

# resonator2 (teal, mirrors resonator1 going rightward)
ax2.fill(*res2_poly.exterior.xy, alpha=0.55, color="teal", zorder=2)
ax2.plot(*res2_poly.exterior.xy, color="teal", lw=1.5, zorder=3)
cx, cy = res2_poly.centroid.x, res2_poly.centroid.y
ax2.text(cx, cy, "resonator2", ha="center", va="center", fontsize=7,
    bbox=dict(boxstyle="round,pad=0.2", fc="teal", alpha=0.8, ec="none"),
    color="white", zorder=5)

ax2.autoscale_view(); ax2.set_aspect("equal")
ax2.set_title(
    "Extended Topology (6 components)\n"
    "qubit → claw → resonator1 → feedline (upper)\n"
    "                           resonator2 ← feedline (lower)",
    fontweight="bold", fontsize=9)
ax2.grid(True, alpha=0.3)
ax2.set_xlabel("x (μm)"); ax2.set_ylabel("y (μm)")
plt.tight_layout()
plt.show()


In [ ]:
ext_netlist = CircuitNetlist(
    components=[
        ComponentSpec(name="qubit", component_type="TransmonCross"),
        ComponentSpec(name="claw", component_type="Claw"),
        ComponentSpec(name="resonator1", component_type="RouteMeander"),
        ComponentSpec(name="feedline1", component_type="CoupledLineTee"),
        ComponentSpec(name="feedline2", component_type="CoupledLineTee"),
        ComponentSpec(name="resonator2", component_type="RouteMeander"),
    ],
    edges=[
        EdgeSpec(src="qubit", dst="claw", coupling_type="capacitive"),
        EdgeSpec(src="claw", dst="resonator1", coupling_type="galvanic"),
        EdgeSpec(src="resonator1", dst="feedline1", coupling_type="capacitive"),
        EdgeSpec(src="feedline1", dst="feedline2", coupling_type="galvanic"),
        EdgeSpec(src="feedline2", dst="resonator2", coupling_type="capacitive"),
    ],
)

data_ext = builder.build(lyt_ext, ext_netlist, global_features={"dielectric_constant": 11.45})
with torch.no_grad():
    out_ext = model(data_ext)

print("=== Case 2: Extended Topology (6 components) ===")
print(f"Graph: {data_ext['component'].x.size(0)} components, "
      f"{data_ext['component','physical','component'].edge_index.size(1)//2} physical edges\n")

print("Node predictions (inference readout):")
for i, (name, ctype) in enumerate(zip(data_ext['component'].component_name, data_ext['component'].component_type)):
    readout = NODE_INFERENCE_READOUT.get(ctype, [])
    preds = out_ext['node_preds'][i]
    for j, tname in enumerate(NODE_TARGET_NAMES):
        if tname in readout:
            print(f"  {name:14s} -> {tname}: {preds[j].item() * node_scales[j]:.3f}")

print("\nEdge predictions (inference readout):")
for line in read_edge_predictions(data_ext, out_ext, edge_scales):
    print(line)

print("\nKey observations:")
print("  - resonator2 automatically gets its own cavity_freq prediction")
print("  - feedline2<->resonator2 edge gets its own kappa prediction")
print("  - Each physical connection produces ONE value (averaged over both directions)")


---
## Summary

| | Tabular DNN (Tutorial 8) | Universal GNN (Tutorial 12) |
|---|---|---|
| Input | Fixed parameter vector | Heterogeneous graph of geometric embeddings |
| Node targets | All from one output | `qubit_freq`, `anharmonicity`, `cavity_freq` per component |
| Edge targets | N/A | `g`, `kappa` per physical edge |
| Training | Manual feature engineering | GNN learns correlations through message passing |
| New topology | Impossible | Seamless — just build the graph |
| Readout | Single output | Per-type readout map (scalable) |
